In [4]:
import pandas as pd
from sqlalchemy import create_engine

# Connect to MySQL
engine = create_engine('mysql+mysqlconnector://root:@localhost/ecommerce_db')

# Create database first — run this once
from sqlalchemy import text
with engine.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS ecommerce_db"))
    conn.execute(text("USE ecommerce_db"))

print("Connected!")

ProgrammingError: (mysql.connector.errors.ProgrammingError) 1049 (42000): Unknown database 'ecommerce_db'
(Background on this error at: https://sqlalche.me/e/20/f405)

In [5]:
import pandas as pd
from sqlalchemy import create_engine, text
import mysql.connector

# Step 1: Connect WITHOUT specifying a database
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password=''
)
cursor = conn.cursor()

# Step 2: Create the database
cursor.execute("CREATE DATABASE IF NOT EXISTS ecommerce_db")
print("Database created!")

cursor.close()
conn.close()

Database created!


In [6]:
# Connect to the newly created database
engine = create_engine('mysql+mysqlconnector://root:@localhost/ecommerce_db')

# Test connection
with engine.connect() as c:
    print("Connected to ecommerce_db successfully!")

Connected to ecommerce_db successfully!


In [8]:
# Reset connection first
engine.dispose()
engine = create_engine('mysql+mysqlconnector://root:@localhost/ecommerce_db')

tables = {
    'dim_customers': pd.read_csv('../data/dim_customers.csv'),
    'dim_products':  pd.read_csv('../data/dim_products.csv'),
    'dim_location':  pd.read_csv('../data/dim_location.csv'),
    'dim_shipping':  pd.read_csv('../data/dim_shipping.csv'),
    'fact_orders':   pd.read_csv('../data/fact_orders.csv',
                                  parse_dates=['order_date'])
}

for name, df in tables.items():
    try:
        df.to_sql(name, engine, if_exists='replace', index=False)
        print(f"✅ Loaded {name}: {len(df):,} rows")
    except Exception as e:
        print(f"❌ Failed {name}: {e}")
        engine.dispose()
        engine = create_engine('mysql+mysqlconnector://root:@localhost/ecommerce_db')

print("\nDone!")

✅ Loaded dim_customers: 793 rows
✅ Loaded dim_products: 1,894 rows
✅ Loaded dim_location: 632 rows
✅ Loaded dim_shipping: 5,009 rows
❌ Failed fact_orders: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

Done!


In [9]:
import mysql.connector
from sqlalchemy import create_engine
import pandas as pd

# Step 1 — Force rollback via raw connector
raw = mysql.connector.connect(
    host='localhost', user='root', password='', database='ecommerce_db'
)
raw.rollback()
raw.close()
print("Rollback done!")

# Step 2 — Fresh engine
engine = create_engine(
    'mysql+mysqlconnector://root:@localhost/ecommerce_db',
    pool_pre_ping=True
)

# Step 3 — Load only fact_orders
fact_orders = pd.read_csv('../data/fact_orders.csv', parse_dates=['order_date'])

with engine.begin() as conn:
    fact_orders.to_sql('fact_orders', conn, if_exists='replace', index=False)

print(f"✅ Loaded fact_orders: {len(fact_orders):,} rows")

Rollback done!


Exception during reset or similar
Traceback (most recent call last):
  File "C:\Users\ZAHIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\mysql\connector\connection_cext.py", line 772, in cmd_query
    self._cmysql.query(
    ~~~~~~~~~~~~~~~~~~^
        query,
        ^^^^^^
    ...<3 lines>...
        query_attrs=self.query_attrs,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
_mysql_connector.MySQLInterfaceError: Got a packet bigger than 'max_allowed_packet' bytes

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\ZAHIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sqlalchemy\engine\base.py", line 1936, in _exec_single_context
    self.dialect.do_executemany(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        cursor,
        ^^^^^^^
    ...<2 lines>...
        context,
        ^^^^^^^^
    )
    ^
  File "C:\Users\ZAHIN\AppData\Local\Programs\Python\Python313\Lib\site-packages\sqlalchemy\engine\d

MySQLInterfaceError: Lost connection to MySQL server during query

In [10]:
import mysql.connector
from sqlalchemy import create_engine, text
import pandas as pd

# Step 1 — Increase max_allowed_packet via raw connector
raw = mysql.connector.connect(
    host='localhost', user='root', password='', database='ecommerce_db'
)
cursor = raw.cursor()
cursor.execute("SET GLOBAL max_allowed_packet = 67108864")  # 64MB
raw.commit()
cursor.close()
raw.close()
print("Packet size increased!")

# Step 2 — Fresh engine
engine = create_engine(
    'mysql+mysqlconnector://root:@localhost/ecommerce_db',
    pool_pre_ping=True
)

# Step 3 — Load fact_orders in chunks of 1000
fact_orders = pd.read_csv('../data/fact_orders.csv', 
                           parse_dates=['order_date'])

with engine.begin() as conn:
    fact_orders.to_sql('fact_orders', conn, 
                       if_exists='replace', 
                       index=False,
                       chunksize=1000)

print(f"✅ Loaded fact_orders: {len(fact_orders):,} rows")

Packet size increased!
✅ Loaded fact_orders: 9,994 rows


In [11]:
def run_query(sql):
    return pd.read_sql(sql, engine)

print("Ready!")

Ready!


In [12]:
q1 = """
SELECT 
    p.category,
    COUNT(DISTINCT f.order_id)      AS total_orders,
    ROUND(SUM(f.sales), 2)          AS total_sales,
    ROUND(SUM(f.profit), 2)         AS total_profit,
    ROUND(AVG(f.profit_margin), 2)  AS avg_profit_margin
FROM fact_orders f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY total_sales DESC
"""
df_q1 = run_query(q1)
print(df_q1)

          category  total_orders  total_sales  total_profit  avg_profit_margin
0       Technology          1544    893633.28     153415.70              15.48
1        Furniture          1764    764284.65      20098.89               4.27
2  Office Supplies          3742    736748.59     126113.35              14.02


In [15]:
q2 = """
SELECT 
    l.region,
    c.segment,
    ROUND(SUM(f.sales), 2)    AS total_sales,
    ROUND(SUM(f.profit), 2)   AS total_profit,
    COUNT(DISTINCT f.order_id) AS total_orders
FROM fact_orders f
JOIN dim_customers c ON f.customer_id = c.customer_id
JOIN dim_location  l ON f.postal_code  = l.postal_code
GROUP BY l.region, c.segment
ORDER BY l.region, total_sales DESC
"""
df_q2 = run_query(q2)
print(df_q2)

     region      segment  total_sales  total_profit  total_orders
0   Central     Consumer    252031.43       8564.05           604
1   Central    Corporate    157995.81      18703.90           348
2   Central  Home Office     91212.64      12438.41           223
3      East     Consumer    350908.17      41190.98           713
4      East    Corporate    200409.35      23622.58           434
5      East  Home Office    127463.73      26709.22           254
6     South     Consumer    195580.97      26913.57           444
7     South    Corporate    121885.93      15215.22           247
8     South  Home Office     74255.00       4620.63           131
9      West     Consumer    368146.54      58529.14           825
10     West    Corporate    226667.25      34617.61           485
11     West  Home Office    138290.17      16733.05           301


In [16]:
q3 = """
SELECT 
    p.product_name,
    p.category,
    p.sub_category,
    ROUND(SUM(f.sales), 2)         AS total_sales,
    ROUND(SUM(f.profit), 2)        AS total_profit,
    ROUND(AVG(f.profit_margin), 2) AS avg_margin,
    SUM(f.quantity)                AS units_sold
FROM fact_orders f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY p.product_name, p.category, p.sub_category
ORDER BY total_profit DESC
LIMIT 10
"""
df_q3 = run_query(q3)
print(df_q3)

                                        product_name         category  \
0              Canon imageCLASS 2200 Advanced Copier       Technology   
1  Fellowes PB500 Electric Punch Plastic Comb Bin...  Office Supplies   
2               Hewlett Packard LaserJet 3310 Copier       Technology   
3                 Canon PC1060 Personal Laser Copier       Technology   
4          Logitech G19 Programmable Gaming Keyboard       Technology   
5  Plantronics Savi W720 Multi-Device Wireless He...       Technology   
6  HP Designjet T520 Inkjet Large Format Printer ...       Technology   
7                  Ativa V4110MDD Micro-Cut Shredder       Technology   
8   3D Systems Cube Printer, 2nd Generation, Magenta       Technology   
9               Ibico EPK-21 Electric Binding System  Office Supplies   

  sub_category  total_sales  total_profit  avg_margin  units_sold  
0      Copiers     61599.82      25199.93       38.47        20.0  
1      Binders     27453.38       7753.04        5.00       

In [17]:
q4 = """
SELECT 
    p.product_name,
    p.category,
    p.sub_category,
    ROUND(SUM(f.profit), 2)        AS total_profit,
    ROUND(AVG(f.discount), 2)      AS avg_discount,
    ROUND(AVG(f.profit_margin), 2) AS avg_margin,
    COUNT(*)                       AS order_count
FROM fact_orders f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY p.product_name, p.category, p.sub_category
HAVING total_profit < 0
ORDER BY total_profit ASC
LIMIT 10
"""
df_q4 = run_query(q4)
print(df_q4)

                                        product_name         category  \
0          Cubify CubeX 3D Printer Double Head Print       Technology   
1          Lexmark MX611dhe Monochrome Laser Printer       Technology   
2          Cubify CubeX 3D Printer Triple Head Print       Technology   
3  Chromcraft Bull-Nose Wood Oval Conference Tabl...        Furniture   
4  Bush Advantage Collection Racetrack Conference...        Furniture   
5          GBC DocuBind P400 Electric Binding System  Office Supplies   
6  Cisco TelePresence System EX90 Videoconferenci...       Technology   
7  Martin Yale Chadless Opener Electric Letter Op...  Office Supplies   
8                       Balt Solid Wood Round Tables        Furniture   
9  BoxOffice By Design Rectangular and Half-Moon ...        Furniture   

  sub_category  total_profit  avg_discount  avg_margin  order_count  
0     Machines      -8879.97          0.53      -95.28            3  
1     Machines      -4589.97          0.40      -36.11  

In [18]:
q5 = """
SELECT 
    order_year,
    order_month,
    ROUND(SUM(sales), 2)   AS monthly_sales,
    ROUND(SUM(profit), 2)  AS monthly_profit,
    COUNT(DISTINCT order_id) AS orders
FROM fact_orders
GROUP BY order_year, order_month
ORDER BY order_year, order_month
"""
df_q5 = run_query(q5)
print(df_q5.head(20))

    order_year  order_month  monthly_sales  monthly_profit  orders
0         2014            1       14236.89         2450.19      32
1         2014            2        4519.89          862.31      28
2         2014            3       55691.01          498.73      71
3         2014            4       28295.34         3488.84      66
4         2014            5       23648.29         2738.71      69
5         2014            6       34595.13         4976.52      66
6         2014            7       33946.39         -841.48      65
7         2014            8       27909.47         5318.10      72
8         2014            9       81777.35         8328.10     130
9         2014           10       31453.39         3448.26      78
10        2014           11       78628.72         9292.13     151
11        2014           12       69545.62         8983.57     141
12        2015            1       18174.08        -3281.01      29
13        2015            2       11951.41         2813.85    

In [19]:
q6 = """
SELECT 
    c.customer_name,
    c.segment,
    ROUND(SUM(f.sales), 2) AS total_sales,
    ROUND(SUM(f.profit), 2) AS total_profit,
    COUNT(DISTINCT f.order_id) AS total_orders,
    RANK() OVER (ORDER BY SUM(f.sales) DESC) AS sales_rank
FROM fact_orders f
JOIN dim_customers c ON f.customer_id = c.customer_id
GROUP BY c.customer_name, c.segment
ORDER BY sales_rank
LIMIT 15
"""
df_q6 = run_query(q6)
print(df_q6)

         customer_name      segment  total_sales  total_profit  total_orders  \
0          Sean Miller  Home Office     25043.05      -1980.74             5   
1         Tamara Chand    Corporate     19052.22       8981.32             5   
2         Raymond Buch     Consumer     15117.34       6976.10             6   
3         Tom Ashbrook  Home Office     14595.62       4703.79             4   
4        Adrian Barton     Consumer     14473.57       5444.81            10   
5         Ken Lonsdale     Consumer     14175.23        806.86            12   
6         Sanjit Chand     Consumer     14142.33       5757.41             9   
7         Hunter Lopez     Consumer     12873.30       5622.43             6   
8         Sanjit Engle     Consumer     12209.44       2650.68            11   
9   Christopher Conant     Consumer     12129.07       2177.05             5   
10        Todd Sumrall    Corporate     11891.75       2371.71             6   
11           Greg Tran     Consumer     

In [20]:
q7 = """
SELECT 
    s.ship_mode,
    COUNT(*) AS total_shipments,
    ROUND(AVG(f.shipping_days), 1) AS avg_ship_days,
    ROUND(SUM(f.sales), 2) AS total_sales,
    ROUND(SUM(f.profit), 2) AS total_profit
FROM fact_orders f
JOIN dim_shipping s ON f.order_id = s.order_id
GROUP BY s.ship_mode
ORDER BY avg_ship_days
"""
df_q7 = run_query(q7)
print(df_q7)

        ship_mode  total_shipments  avg_ship_days  total_sales  total_profit
0        Same Day              543            0.0    128363.12      15891.76
1     First Class             1538            2.2    351428.42      48969.84
2    Second Class             1945            3.2    459193.57      57446.64
3  Standard Class             5968            5.0   1358215.74     164088.79


In [21]:
q8 = """
SELECT 
    CASE 
        WHEN discount = 0         THEN 'No Discount'
        WHEN discount <= 0.10     THEN 'Low (1-10%)'
        WHEN discount <= 0.30     THEN 'Medium (11-30%)'
        WHEN discount <= 0.50     THEN 'High (31-50%)'
        ELSE 'Very High (50%+)'
    END AS discount_tier,
    COUNT(*) AS order_count,
    ROUND(AVG(profit_margin), 2) AS avg_profit_margin,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(AVG(sales), 2) AS avg_order_value
FROM fact_orders
GROUP BY discount_tier
ORDER BY avg_profit_margin DESC
"""
df_q8 = run_query(q8)
print(df_q8)

      discount_tier  order_count  avg_profit_margin  total_profit  \
0       No Discount         4798              34.02     320987.60   
1   Medium (11-30%)         3936              15.81      81387.02   
2       Low (1-10%)           94              15.58       9029.18   
3     High (31-50%)          310             -29.61     -48447.73   
4  Very High (50%+)          856            -113.88     -76559.05   

   avg_order_value  
0           226.74  
1           227.48  
2           578.40  
3           630.05  
4            75.03  


In [22]:
q9 = """
SELECT 
    l.state,
    l.region,
    COUNT(DISTINCT f.order_id) AS total_orders,
    ROUND(SUM(f.sales), 2) AS total_sales,
    ROUND(SUM(f.profit), 2) AS total_profit,
    ROUND(AVG(f.profit_margin), 2) AS avg_margin
FROM fact_orders f
JOIN dim_location l ON f.postal_code = l.postal_code
GROUP BY l.state, l.region
ORDER BY total_sales DESC
LIMIT 15
"""
df_q9 = run_query(q9)
print(df_q9)

             state   region  total_orders  total_sales  total_profit  \
0       California     West          1021    465333.76      77842.74   
1         New York     East           562    310876.27      74038.55   
2            Texas  Central           487    170188.05     -25729.36   
3       Washington     West           256    138641.27      33402.65   
4     Pennsylvania     East           288    116511.91     -15559.96   
5          Florida    South           200     89473.71      -3399.30   
6         Illinois  Central           276     80166.10     -12607.89   
7             Ohio     East           236     78258.14     -16971.38   
8         Michigan  Central           117     76269.61      24463.19   
9         Virginia    South           115     70636.72      18597.95   
10  North Carolina    South           136     55603.16      -7490.91   
11         Indiana  Central            73     53555.36      18382.94   
12         Georgia    South            91     49095.84      1625